# Phase 1B Silver Transformation

This notebook executes the transaction-safe Silver and Audit transformation against the existing public source schema. It preserves source rows and does not create Gold tables.


In [1]:
from __future__ import annotations

import csv
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import psycopg2
from dotenv import load_dotenv


ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / ".env").exists():
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

connection = psycopg2.connect(
    host=os.environ["PGHOST"],
    port=os.environ["PGPORT"],
    dbname=os.environ["PGDATABASE"],
    user=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
)
connection.autocommit = False
audit_connection = psycopg2.connect(
    host=os.environ["PGHOST"],
    port=os.environ["PGPORT"],
    dbname=os.environ["PGDATABASE"],
    user=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
)
audit_connection.autocommit = True


def query(sql: str, params: tuple = ()) -> list[dict]:
    with connection.cursor() as cursor:
        cursor.execute(sql, params)
        columns = [item[0] for item in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]


def run(sql: str, params: tuple = ()) -> None:
    with connection.cursor() as cursor:
        cursor.execute(sql, params)


def audit_run(sql: str, params: tuple = ()) -> None:
    with audit_connection.cursor() as cursor:
        cursor.execute(sql, params)


audit_run(
    """
    CREATE SCHEMA IF NOT EXISTS audit;
    CREATE TABLE IF NOT EXISTS audit.transformation_run (
      run_id uuid PRIMARY KEY, started_at timestamptz NOT NULL, completed_at timestamptz,
      status text NOT NULL, source_procurement_rows bigint, silver_procurement_rows bigint,
      source_rule_register text, validation_summary jsonb, error_message text
    )
    """
)

source = query(
    """
    SELECT
      (SELECT COUNT(*) FROM public.fact_procurement_monthly) AS procurement_rows,
      (SELECT COUNT(DISTINCT procurement_id) FROM public.fact_procurement_monthly) AS procurement_ids,
      (SELECT COUNT(*) FROM public.fact_vendor_performance_quarterly) AS performance_rows,
      (SELECT COUNT(*) FROM public.bridge_vendor_material) AS relationship_rows,
      (SELECT COUNT(*) FROM public.bridge_material_commodity) AS commodity_mapping_rows,
      (SELECT COUNT(*) FROM public.fact_commodity_monthly) AS commodity_rows,
      (SELECT COUNT(*) FROM public.fact_fx_monthly) AS fx_rows
    """
)[0]
rule_path = ROOT / "data_report/reports/phase1_transformation_rule_register.csv"
with rule_path.open(encoding="utf-8") as handle:
    rules = list(csv.DictReader(handle))

pilot_ids = [
    "POA0002323",
    "POA10002323",
    "POA0001056",
    "POA0001265",
    "POA0000247",
    "POA0001364",
    "POA0002355",
    "POA0000591",
    "POA0000311",
    "POA0001292",
]
pilots = query(
    """
    SELECT p.procurement_id, p.vendor_material_id,
           p.vendor_id AS reported_vendor_id, b.vendor_id AS canonical_vendor_id,
           p.material_id, p.plant_id, p.month, p.order_currency,
           p.order_value_local, p.po_qty, p.po_uom, m.base_uom,
           b.contract_currency
    FROM public.fact_procurement_monthly p
    LEFT JOIN public.bridge_vendor_material b
      ON b.vendor_material_id = p.vendor_material_id
    LEFT JOIN public.dim_material m ON m.material_id = p.material_id
    WHERE p.procurement_id = ANY(%s)
    ORDER BY p.procurement_id
    """,
    (pilot_ids,),
)
multi_plant = query(
    """
    SELECT vendor_id, material_id
    FROM public.bridge_vendor_material
    GROUP BY vendor_id, material_id
    HAVING COUNT(DISTINCT plant_id) > 1
    """
)
assert len(pilots) == 10
assert not multi_plant
missing_month_check = query(
    """
    WITH bounds AS (
      SELECT min(month) AS lo, max(month) AS hi
      FROM public.fact_procurement_monthly
    ), expected AS (
      SELECT vm.vendor_material_id, gs::date AS procurement_month
      FROM public.bridge_vendor_material vm
      CROSS JOIN bounds
      CROSS JOIN LATERAL generate_series(
        date_trunc('month', COALESCE(vm.approved_from, bounds.lo)),
        date_trunc('month', COALESCE(vm.approved_to, bounds.hi)),
        interval '1 month'
      ) gs
      WHERE vm.vendor_material_id = 'VM00011'
    )
    SELECT e.vendor_material_id, e.procurement_month
    FROM expected e
    LEFT JOIN public.fact_procurement_monthly p
      ON p.vendor_material_id = e.vendor_material_id
     AND p.month = e.procurement_month
    WHERE p.procurement_id IS NULL
    ORDER BY e.procurement_month
    LIMIT 1
    """
)
assert missing_month_check
prior_uom_signature = query(
    """
    SELECT COALESCE(string_agg(
      concat_ws('|', from_uom, to_uom, conversion_factor::text,
                effective_from::text, effective_to::text, approval_status,
                COALESCE(source_reference, '')), ',' ORDER BY from_uom, to_uom,
                effective_from, effective_to, approval_status, source_reference), '') AS signature
    FROM silver.uom_conversion
    """
)[0]["signature"]
prior_duplicate_runs = query(
    "SELECT COUNT(DISTINCT run_id) AS count FROM audit.procurement_duplicate_review"
)[0]["count"]
print(
    {
        "database": os.environ["PGDATABASE"],
        "source": source,
        "rule_count": len(rules),
        "pilot_rows": len(pilots),
        "multi_plant_pairs": 0,
    }
)

run_id = str(uuid.uuid4())
started = datetime.now(timezone.utc)
audit_run(
    """
    INSERT INTO audit.transformation_run
      (run_id, started_at, status, source_rule_register)
    VALUES (%s, %s, 'RUNNING', 'phase1_transformation_rule_register.csv')
    """,
    (run_id, started),
)

try:
    run("CREATE SCHEMA IF NOT EXISTS silver")
    run("CREATE SCHEMA IF NOT EXISTS audit")
    run(
        """
        CREATE TABLE IF NOT EXISTS silver.vendor AS TABLE public.dim_vendor WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.material AS TABLE public.dim_material WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.plant AS TABLE public.dim_plant WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.purchasing_org AS TABLE public.dim_purchasing_org WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.commodity AS TABLE public.dim_commodity WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.fx_monthly AS TABLE public.fact_fx_monthly WITH NO DATA;
        CREATE TABLE IF NOT EXISTS silver.uom_conversion (
          from_uom text NOT NULL, to_uom text NOT NULL, conversion_factor numeric NOT NULL,
          effective_from date, effective_to date, approval_status text NOT NULL,
          source_reference text, PRIMARY KEY (from_uom, to_uom, effective_from)
        );
        CREATE TABLE IF NOT EXISTS silver.procurement_relationship (
          vendor_material_id text PRIMARY KEY, vendor_id text NOT NULL, material_id text NOT NULL,
          plant_id text NOT NULL, contract_id text, approved_from date, approved_to date,
          contract_currency text, quoted_unit_price numeric, lead_time_days integer,
          min_order_qty integer, allocation_target_pct numeric
        );
        CREATE TABLE IF NOT EXISTS silver.procurement_clean (
          source_procurement_id text PRIMARY KEY, vendor_material_id text, reported_vendor_id text,
          canonical_vendor_id text, vendor_resolution_status text NOT NULL, source_material_id text,
          source_plant_id text, purch_org_id text, procurement_month date, material_description text,
          material_group_id text, source_system text, document_grain text, order_value_local numeric,
          original_order_currency text, normalized_currency text, original_po_qty numeric,
          original_po_uom text, original_gr_qty numeric, original_gr_uom text,
          source_unit_price_local numeric, source_order_value_eur numeric, material_base_uom text,
          contract_currency text, approved_from date, approved_to date, applied_units_per_eur numeric,
          applied_fx_month date, applied_fx_rate_type text, applied_fx_source_label text,
          fx_conversion_status text NOT NULL, po_to_base_factor numeric, standardized_po_qty numeric,
          uom_conversion_status text NOT NULL, eur_expenditure numeric, eur_unit_price numeric,
          price_reconciliation_status text NOT NULL, duplicate_group_id text,
          duplicate_investigation_status text NOT NULL, zero_quantity_positive_spend_flag boolean NOT NULL,
          missing_quantity_flag boolean NOT NULL, missing_value_flag boolean NOT NULL,
          negative_gr_flag boolean NOT NULL, gr_exceeds_po_flag boolean NOT NULL,
          uom_mismatch_flag boolean NOT NULL, material_active_period_flag boolean NOT NULL,
          relationship_validity_flag boolean NOT NULL, exception_count integer NOT NULL
        );
        CREATE INDEX IF NOT EXISTS ix_procurement_clean_vmm
          ON silver.procurement_clean(vendor_material_id, procurement_month);
        CREATE TABLE IF NOT EXISTS silver.material_commodity_monthly (
          material_id text NOT NULL, commodity_id text NOT NULL, month date NOT NULL,
          exposure_weight numeric, index_price numeric, weighted_index_contribution numeric,
          index_price_missing_flag boolean NOT NULL, weight_validation_status text NOT NULL,
          PRIMARY KEY (material_id, commodity_id, month)
        );
        CREATE TABLE IF NOT EXISTS audit.transformation_run (
          run_id uuid PRIMARY KEY, started_at timestamptz NOT NULL, completed_at timestamptz,
          status text NOT NULL, source_procurement_rows bigint, silver_procurement_rows bigint,
          source_rule_register text, validation_summary jsonb, error_message text
        );
        CREATE TABLE IF NOT EXISTS audit.procurement_exceptions (
          exception_id bigserial PRIMARY KEY, run_id uuid NOT NULL REFERENCES audit.transformation_run(run_id),
          source_table text NOT NULL, source_record_id text, rule_id text NOT NULL,
          exception_category text NOT NULL, severity text NOT NULL, reason text NOT NULL,
          original_context jsonb, resolution_status text NOT NULL, detected_at timestamptz NOT NULL
        );
        CREATE TABLE IF NOT EXISTS audit.procurement_duplicate_review (
          duplicate_group_id text NOT NULL, run_id uuid NOT NULL REFERENCES audit.transformation_run(run_id),
          vendor_material_id text NOT NULL, procurement_month date NOT NULL,
          source_procurement_ids text[] NOT NULL, group_row_count integer NOT NULL,
          local_value_sum numeric, po_qty_sum numeric, investigation_status text NOT NULL,
          reconciliation_status text NOT NULL, pending_business_decision text NOT NULL
        );
        CREATE TABLE IF NOT EXISTS audit.performance_duplicate_review (
          review_id bigserial PRIMARY KEY, run_id uuid NOT NULL REFERENCES audit.transformation_run(run_id),
          vendor_id text NOT NULL, plant_id text NOT NULL, quarter_start date NOT NULL,
          source_row_count integer NOT NULL, source_row_ids text[] NOT NULL,
          delivery_lines_sum numeric, delivery_lines_min numeric, delivery_lines_max numeric,
          investigation_status text NOT NULL, weighting_status text NOT NULL,
          pending_business_decision text NOT NULL,
          UNIQUE (run_id, vendor_id, plant_id, quarter_start)
        );
        CREATE TABLE IF NOT EXISTS audit.price_reconciliation_review (
          run_id uuid NOT NULL REFERENCES audit.transformation_run(run_id),
          source_procurement_id text NOT NULL,
          source_unit_price_local numeric,
          derived_unit_price_local numeric,
          absolute_difference numeric,
          relative_difference numeric,
          reconciliation_status text NOT NULL,
          investigation_note text NOT NULL,
          PRIMARY KEY (run_id, source_procurement_id)
        )
        """
    )
    run("ALTER TABLE audit.procurement_duplicate_review DROP CONSTRAINT IF EXISTS procurement_duplicate_review_pkey")
    run(
        """
        DO $$
        BEGIN
          IF NOT EXISTS (
            SELECT 1 FROM pg_constraint
            WHERE conrelid = 'audit.procurement_duplicate_review'::regclass
              AND conname = 'procurement_duplicate_review_pk'
          ) THEN
            ALTER TABLE audit.procurement_duplicate_review
              ADD CONSTRAINT procurement_duplicate_review_pk
              PRIMARY KEY (run_id, duplicate_group_id);
          END IF;
        END $$;
        CREATE INDEX IF NOT EXISTS ix_duplicate_review_group
          ON audit.procurement_duplicate_review(duplicate_group_id);
        """
    )
    run(
        """
        CREATE TEMP TABLE stage_procurement ON COMMIT DROP AS
        WITH duplicate_groups AS (
          SELECT vendor_material_id, month,
                 md5(vendor_material_id || '|' || month::text) AS duplicate_group_id
          FROM public.fact_procurement_monthly
          GROUP BY vendor_material_id, month
          HAVING COUNT(*) > 1
        ), base AS (
          SELECT p.*, vm.vendor_id AS canonical_vendor_id,
                 vm.material_id AS bridge_material_id, vm.plant_id AS bridge_plant_id,
                 vm.contract_currency, vm.approved_from, vm.approved_to,
                 m.base_uom, m.active_from, m.active_to,
                 fx.units_per_eur, fx.month AS fx_month, fx.rate_type, fx.source_label,
                 dg.duplicate_group_id
          FROM public.fact_procurement_monthly p
          LEFT JOIN public.bridge_vendor_material vm
            ON vm.vendor_material_id = p.vendor_material_id
          LEFT JOIN public.dim_material m ON m.material_id = p.material_id
          LEFT JOIN public.fact_fx_monthly fx
            ON fx.from_currency = 'EUR'
           AND fx.to_currency = p.order_currency
           AND fx.month = p.month
          LEFT JOIN duplicate_groups dg
            ON dg.vendor_material_id = p.vendor_material_id AND dg.month = p.month
        )
        SELECT procurement_id AS source_procurement_id, vendor_material_id,
          vendor_id AS reported_vendor_id, canonical_vendor_id,
          CASE WHEN canonical_vendor_id IS NULL THEN 'UNRESOLVED'
               WHEN vendor_id = canonical_vendor_id THEN 'MATCH' ELSE 'MISMATCH' END AS vendor_resolution_status,
          material_id AS source_material_id, plant_id AS source_plant_id, purch_org_id,
          month AS procurement_month, material_description, material_group_id, source_system, document_grain,
          order_value_local, order_currency AS original_order_currency,
          CASE WHEN order_currency = 'EUR' THEN 'EUR' ELSE order_currency END AS normalized_currency,
          po_qty AS original_po_qty, po_uom AS original_po_uom,
          gr_qty AS original_gr_qty, gr_uom AS original_gr_uom,
          unit_price_local AS source_unit_price_local, order_value_eur AS source_order_value_eur,
          base_uom AS material_base_uom, contract_currency, approved_from, approved_to,
          CASE WHEN order_currency = 'EUR' THEN 1::numeric
               WHEN order_currency = 'EURO' THEN NULL ELSE units_per_eur END AS applied_units_per_eur,
          CASE WHEN order_currency IN ('EUR', 'EURO') THEN NULL ELSE fx_month END AS applied_fx_month,
          CASE WHEN order_currency = 'EUR' THEN 'IDENTITY_RATE'
               WHEN order_currency = 'EURO' THEN 'ALIAS_PENDING'
               ELSE 'MONTHLY_AVERAGE_PROXY' END AS applied_fx_rate_type,
          CASE WHEN order_currency = 'EUR' THEN 'EUR_IDENTITY'
               WHEN order_currency = 'EURO' THEN 'EURO_ALIAS_PENDING'
               WHEN source_label IS NULL THEN NULL
               ELSE 'MONTHLY_FX_PROXY:' || source_label END AS applied_fx_source_label,
          CASE WHEN order_currency = 'EURO' THEN 'ALIAS_PENDING'
               WHEN order_currency = 'EUR' THEN 'EUR_IDENTITY_RATE'
               WHEN units_per_eur IS NULL THEN 'MISSING_RATE'
               ELSE 'APPLIED_MONTHLY_PROXY' END AS fx_conversion_status,
          CASE WHEN po_uom = base_uom AND base_uom IS NOT NULL THEN 1::numeric ELSE NULL END AS po_to_base_factor,
          CASE WHEN po_uom = base_uom AND po_qty IS NOT NULL THEN po_qty ELSE NULL END AS standardized_po_qty,
          CASE WHEN po_uom = base_uom AND base_uom IS NOT NULL THEN 'IDENTITY_VERIFIED'
               WHEN po_uom IS NULL OR base_uom IS NULL THEN 'UNRESOLVED'
               ELSE 'PENDING_APPROVAL' END AS uom_conversion_status,
          CASE WHEN order_value_local IS NULL OR order_currency = 'EURO' THEN NULL
               WHEN order_currency = 'EUR' THEN order_value_local
               WHEN units_per_eur IS NOT NULL THEN order_value_local / units_per_eur END AS eur_expenditure,
          NULL::numeric AS eur_unit_price, 'PENDING_PRICE_ELIGIBILITY' AS price_reconciliation_status,
          duplicate_group_id,
          CASE WHEN duplicate_group_id IS NULL THEN 'NOT_DUPLICATE' ELSE 'PENDING_REVIEW' END AS duplicate_investigation_status,
          COALESCE(po_qty = 0 AND order_value_local > 0, false) AS zero_quantity_positive_spend_flag,
          (po_qty IS NULL) AS missing_quantity_flag, (order_value_local IS NULL) AS missing_value_flag,
          COALESCE(gr_qty < 0, false) AS negative_gr_flag,
          COALESCE(gr_qty > po_qty AND po_qty IS NOT NULL, false) AS gr_exceeds_po_flag,
          (po_uom IS DISTINCT FROM base_uom) AS uom_mismatch_flag,
          (active_from IS NOT NULL AND (month < active_from OR (active_to IS NOT NULL AND month > active_to))) AS material_active_period_flag,
          COALESCE(canonical_vendor_id IS NOT NULL AND bridge_material_id = material_id
           AND bridge_plant_id = plant_id
           AND (approved_from IS NULL OR month >= approved_from)
           AND (approved_to IS NULL OR month <= approved_to), false) AS relationship_validity_flag,
          0::integer AS exception_count
        FROM base
        """
    )
    run(
        """
        UPDATE stage_procurement
        SET eur_unit_price = CASE WHEN eur_expenditure IS NOT NULL AND standardized_po_qty > 0
                                  THEN eur_expenditure / standardized_po_qty END,
            price_reconciliation_status = CASE
              WHEN order_value_local IS NULL THEN 'MISSING_VALUE'
              WHEN fx_conversion_status IN ('ALIAS_PENDING', 'MISSING_RATE') THEN 'FX_PENDING'
              WHEN standardized_po_qty IS NULL THEN 'UOM_OR_QUANTITY_PENDING'
              WHEN standardized_po_qty <= 0 THEN 'NON_POSITIVE_QUANTITY'
              ELSE 'ELIGIBLE' END
        """
    )
    staged = query(
        """
        SELECT COUNT(*) AS rows, COUNT(DISTINCT source_procurement_id) AS ids,
               COUNT(*) FILTER (WHERE duplicate_group_id IS NOT NULL) AS duplicate_rows,
               COUNT(*) FILTER (WHERE vendor_resolution_status = 'MISMATCH') AS vendor_mismatches,
               COUNT(*) FILTER (WHERE fx_conversion_status = 'ALIAS_PENDING') AS euro_alias_rows,
               COUNT(*) FILTER (WHERE uom_conversion_status = 'PENDING_APPROVAL') AS uom_pending_rows
        FROM stage_procurement
        """
    )[0]
    assert staged["rows"] == source["procurement_rows"]
    assert staged["ids"] == source["procurement_ids"]

    for table in (
        "material_commodity_monthly", "procurement_clean", "procurement_relationship",
        "fx_monthly", "commodity", "plant", "material", "vendor", "purchasing_org",
    ):
        run(f"DELETE FROM silver.{table}")
    for target, source_table in (
        ("purchasing_org", "dim_purchasing_org"), ("vendor", "dim_vendor"),
        ("material", "dim_material"), ("plant", "dim_plant"), ("commodity", "dim_commodity"),
        ("fx_monthly", "fact_fx_monthly"),
    ):
        run(f"INSERT INTO silver.{target} SELECT * FROM public.{source_table}")
    run(
        """
        INSERT INTO silver.procurement_relationship
        SELECT vendor_material_id, vendor_id, material_id, plant_id, contract_id,
               approved_from, approved_to, contract_currency, quoted_unit_price,
               lead_time_days, min_order_qty, allocation_target_pct
        FROM public.bridge_vendor_material
        """
    )
    run("INSERT INTO silver.procurement_clean SELECT * FROM stage_procurement")
    run(
        """
        INSERT INTO silver.material_commodity_monthly
        WITH mapping_status AS (
          SELECT material_id,
                 CASE WHEN SUM(exposure_weight) BETWEEN .999999 AND 1.000001
                      THEN 'VALIDATED' ELSE 'PENDING_REVIEW' END AS weight_validation_status
          FROM public.bridge_material_commodity
          GROUP BY material_id
        )
        SELECT b.material_id, b.commodity_id, f.month, b.exposure_weight, f.index_price,
               b.exposure_weight * f.index_price, (f.index_price IS NULL),
               ms.weight_validation_status
        FROM public.bridge_material_commodity b
        JOIN public.fact_commodity_monthly f ON f.commodity_id = b.commodity_id
        JOIN mapping_status ms ON ms.material_id = b.material_id
        """
    )
    run(
        """
        DO $$
        BEGIN
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'vendor_pk') THEN
            ALTER TABLE silver.vendor ADD CONSTRAINT vendor_pk PRIMARY KEY (vendor_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'material_pk') THEN
            ALTER TABLE silver.material ADD CONSTRAINT material_pk PRIMARY KEY (material_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'plant_pk') THEN
            ALTER TABLE silver.plant ADD CONSTRAINT plant_pk PRIMARY KEY (plant_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'purchasing_org_pk') THEN
            ALTER TABLE silver.purchasing_org ADD CONSTRAINT purchasing_org_pk PRIMARY KEY (purch_org_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'commodity_pk') THEN
            ALTER TABLE silver.commodity ADD CONSTRAINT commodity_pk PRIMARY KEY (commodity_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'fx_monthly_pk') THEN
            ALTER TABLE silver.fx_monthly
              ADD CONSTRAINT fx_monthly_pk PRIMARY KEY (month, from_currency, to_currency);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'plant_purchasing_org_fk') THEN
            ALTER TABLE silver.plant ADD CONSTRAINT plant_purchasing_org_fk
              FOREIGN KEY (purch_org_id) REFERENCES silver.purchasing_org(purch_org_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'relationship_vendor_fk') THEN
            ALTER TABLE silver.procurement_relationship ADD CONSTRAINT relationship_vendor_fk
              FOREIGN KEY (vendor_id) REFERENCES silver.vendor(vendor_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'relationship_material_fk') THEN
            ALTER TABLE silver.procurement_relationship ADD CONSTRAINT relationship_material_fk
              FOREIGN KEY (material_id) REFERENCES silver.material(material_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'relationship_plant_fk') THEN
            ALTER TABLE silver.procurement_relationship ADD CONSTRAINT relationship_plant_fk
              FOREIGN KEY (plant_id) REFERENCES silver.plant(plant_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'clean_relationship_fk') THEN
            ALTER TABLE silver.procurement_clean ADD CONSTRAINT clean_relationship_fk
              FOREIGN KEY (vendor_material_id)
              REFERENCES silver.procurement_relationship(vendor_material_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'clean_vendor_fk') THEN
            ALTER TABLE silver.procurement_clean ADD CONSTRAINT clean_vendor_fk
              FOREIGN KEY (canonical_vendor_id) REFERENCES silver.vendor(vendor_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'clean_material_fk') THEN
            ALTER TABLE silver.procurement_clean ADD CONSTRAINT clean_material_fk
              FOREIGN KEY (source_material_id) REFERENCES silver.material(material_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'clean_plant_fk') THEN
            ALTER TABLE silver.procurement_clean ADD CONSTRAINT clean_plant_fk
              FOREIGN KEY (source_plant_id) REFERENCES silver.plant(plant_id);
          END IF;
          IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'clean_purchasing_org_fk') THEN
            ALTER TABLE silver.procurement_clean ADD CONSTRAINT clean_purchasing_org_fk
              FOREIGN KEY (purch_org_id) REFERENCES silver.purchasing_org(purch_org_id);
          END IF;
        END $$;
        CREATE INDEX IF NOT EXISTS ix_vendor_name ON silver.vendor(vendor_name);
        CREATE INDEX IF NOT EXISTS ix_material_group ON silver.material(material_group_id);
        CREATE INDEX IF NOT EXISTS ix_plant_purch_org ON silver.plant(purch_org_id);
        CREATE INDEX IF NOT EXISTS ix_relationship_material_plant
          ON silver.procurement_relationship(material_id, plant_id);
        CREATE INDEX IF NOT EXISTS ix_fx_currency_month
          ON silver.fx_monthly(to_currency, month);
        CREATE INDEX IF NOT EXISTS ix_commodity_month
          ON silver.material_commodity_monthly(commodity_id, month);
        """
    )
    run(
        """
        INSERT INTO audit.procurement_duplicate_review
        SELECT duplicate_group_id, %s, vendor_material_id, procurement_month,
               ARRAY_AGG(source_procurement_id ORDER BY source_procurement_id), COUNT(*),
               SUM(order_value_local), SUM(original_po_qty), 'PENDING_REVIEW',
               'NOT_AGGREGATED', 'Business approval required before Gold aggregation'
        FROM stage_procurement
        WHERE duplicate_group_id IS NOT NULL
        GROUP BY duplicate_group_id, vendor_material_id, procurement_month
        ON CONFLICT DO NOTHING
        """,
        (run_id,),
    )
    run(
        """
        INSERT INTO audit.performance_duplicate_review
          (run_id, vendor_id, plant_id, quarter_start, source_row_count, source_row_ids,
           delivery_lines_sum, delivery_lines_min, delivery_lines_max,
           investigation_status, weighting_status, pending_business_decision)
        SELECT %s, vendor_id, plant_id, quarter_start, COUNT(*),
               ARRAY_AGG(performance_id::text ORDER BY performance_id),
               SUM(delivery_lines), MIN(delivery_lines), MAX(delivery_lines),
               'PENDING_REVIEW', 'NOT_APPLIED',
               'Approve delivery_lines weighting and repeated-row reconciliation'
        FROM public.fact_vendor_performance_quarterly
        GROUP BY vendor_id, plant_id, quarter_start
        HAVING COUNT(*) > 1
        ON CONFLICT (run_id, vendor_id, plant_id, quarter_start) DO NOTHING
        """,
        (run_id,),
    )
    run(
        """
        INSERT INTO audit.procurement_exceptions
          (run_id, source_table, source_record_id, rule_id, exception_category,
           severity, reason, original_context, resolution_status, detected_at)
        SELECT %s, 'public.fact_procurement_monthly', source_procurement_id,
               x.rule_id, x.category, x.severity, x.reason,
               jsonb_build_object('vendor_material_id', vendor_material_id,
                 'month', procurement_month, 'order_currency', original_order_currency,
                 'po_uom', original_po_uom, 'material_base_uom', material_base_uom,
                 'order_value_local', order_value_local, 'original_po_qty', original_po_qty),
               'OPEN', clock_timestamp()
        FROM stage_procurement s
        CROSS JOIN LATERAL (VALUES
          ('DQ01','MISSING_ORDER_VALUE','HIGH','Order value is NULL',missing_value_flag),
          ('DQ02','MISSING_OR_NONPOSITIVE_QUANTITY','HIGH','PO quantity is NULL or non-positive',missing_quantity_flag OR original_po_qty <= 0),
          ('DQ04','NEGATIVE_GR','MEDIUM','GR quantity is negative',negative_gr_flag),
          ('DQ06','EURO_ALIAS_PENDING','HIGH','EURO alias approval pending',fx_conversion_status = 'ALIAS_PENDING'),
          ('DQ07','VENDOR_MISMATCH','HIGH','Reported vendor differs from canonical vendor',vendor_resolution_status = 'MISMATCH'),
          ('DQ08','DUPLICATE_CANDIDATE_KEY','HIGH','Repeated candidate key retained for review',duplicate_group_id IS NOT NULL),
          ('DQ09','UOM_MISMATCH','HIGH','PO UOM differs from material base UOM',uom_mismatch_flag),
          ('DQ03','GR_EXCEEDS_PO','MEDIUM','GR quantity exceeds PO quantity',gr_exceeds_po_flag),
          ('RELATIONSHIP','RELATIONSHIP_INVALID','HIGH','Relationship validity check failed',NOT relationship_validity_flag)
        ) x(rule_id, category, severity, reason, applies)
        WHERE applies
        """,
        (run_id,),
    )
    run(
        """
        INSERT INTO audit.price_reconciliation_review
          (run_id, source_procurement_id, source_unit_price_local,
           derived_unit_price_local, absolute_difference, relative_difference,
           reconciliation_status, investigation_note)
        SELECT %s, source_procurement_id, source_unit_price_local,
               CASE WHEN order_value_local IS NOT NULL AND original_po_qty > 0
                    THEN order_value_local / original_po_qty END,
               CASE WHEN source_unit_price_local IS NOT NULL
                          AND order_value_local IS NOT NULL AND original_po_qty > 0
                    THEN ABS(source_unit_price_local - order_value_local / original_po_qty) END,
               CASE WHEN source_unit_price_local IS NOT NULL
                          AND order_value_local IS NOT NULL AND original_po_qty > 0
                          AND source_unit_price_local <> 0
                    THEN ABS(source_unit_price_local - order_value_local / original_po_qty)
                         / ABS(source_unit_price_local) END,
               CASE
                 WHEN source_unit_price_local IS NULL OR order_value_local IS NULL
                      OR original_po_qty IS NULL OR original_po_qty <= 0 THEN 'NOT_TESTABLE'
                 WHEN ABS(source_unit_price_local - order_value_local / original_po_qty)
                      <= GREATEST(.01, ABS(source_unit_price_local) * .0001) THEN 'MATCH'
                 ELSE 'OUTLIER'
               END,
               CASE
                 WHEN source_unit_price_local IS NULL OR order_value_local IS NULL
                      OR original_po_qty IS NULL OR original_po_qty <= 0
                   THEN 'Insufficient source values for reconciliation'
                 ELSE 'Outlier retained for business investigation; source value not overwritten'
               END
        FROM stage_procurement
        ON CONFLICT (run_id, source_procurement_id) DO NOTHING
        """,
        (run_id,),
    )
    run(
        """
        UPDATE silver.procurement_clean p
        SET exception_count = COALESCE(e.exception_count, 0)
        FROM (
          SELECT source_record_id, COUNT(*)::integer AS exception_count
          FROM audit.procurement_exceptions
          WHERE run_id = %s
          GROUP BY source_record_id
        ) e
        WHERE p.source_procurement_id = e.source_record_id
        """,
        (run_id,),
    )
    run(
        """
        UPDATE silver.procurement_clean
        SET exception_count = 0
        WHERE source_procurement_id NOT IN (
          SELECT source_record_id
          FROM audit.procurement_exceptions
          WHERE run_id = %s
        )
        """,
        (run_id,),
    )
    final = query(
        "SELECT COUNT(*) AS rows, COUNT(DISTINCT source_procurement_id) AS ids FROM silver.procurement_clean"
    )[0]
    assert final["rows"] == source["procurement_rows"]
    assert final["ids"] == source["procurement_ids"]
    pilot_actual = query(
        """
        SELECT source_procurement_id, vendor_material_id, source_material_id, source_plant_id,
               order_value_local, original_po_qty, original_order_currency,
               applied_fx_rate_type, fx_conversion_status, uom_conversion_status,
               source_unit_price_local, eur_unit_price, exception_count
        FROM silver.procurement_clean
        WHERE source_procurement_id = ANY(%s)
        """,
        (pilot_ids,),
    )
    pilot_by_id = {row["source_procurement_id"]: row for row in pilot_actual}
    pilot_validation = []
    for expected in pilots:
        actual = pilot_by_id.get(expected["procurement_id"])
        passed = (
            actual is not None
            and actual["vendor_material_id"] == expected["vendor_material_id"]
            and actual["source_material_id"] == expected["material_id"]
            and actual["source_plant_id"] == expected["plant_id"]
            and actual["order_value_local"] == expected["order_value_local"]
            and actual["original_po_qty"] == expected["po_qty"]
            and actual["original_order_currency"] == expected["order_currency"]
        )
        pilot_validation.append((expected["procurement_id"], passed))
    assert len(pilot_validation) == len(pilot_ids)
    assert all(passed for _, passed in pilot_validation)
    silver_missing_month_rows = query(
        """
        SELECT COUNT(*) AS count
        FROM silver.procurement_clean
        WHERE vendor_material_id = %s AND procurement_month = %s
        """,
        (missing_month_check[0]["vendor_material_id"], missing_month_check[0]["procurement_month"]),
    )[0]["count"]
    assert silver_missing_month_rows == 0
    run(
        """
        UPDATE audit.transformation_run
        SET completed_at = %s, status = 'SUCCEEDED',
            source_procurement_rows = %s, silver_procurement_rows = %s,
            validation_summary = %s
        WHERE run_id = %s
        """,
        (
            datetime.now(timezone.utc),
            source["procurement_rows"],
            final["rows"],
            json.dumps({"staged": staged}),
            run_id,
        ),
    )
    connection.commit()
    print({"run_id": str(run_id), "source_rows": source["procurement_rows"], "silver_rows": final["rows"], "staged": staged})
except Exception as exc:
    connection.rollback()
    audit_run(
        """
        UPDATE audit.transformation_run
        SET completed_at = %s, status = 'FAILED', error_message = %s
        WHERE run_id = %s
        """,
        (datetime.now(timezone.utc), str(exc)[:4000], run_id),
    )
    raise

validation = query(
    """
    SELECT
      (SELECT COUNT(*) FROM public.fact_procurement_monthly) AS source_rows,
      (SELECT COUNT(*) FROM silver.procurement_clean) AS silver_rows,
      (SELECT COUNT(DISTINCT source_procurement_id) FROM silver.procurement_clean) AS silver_ids,
      (SELECT COUNT(*) FROM public.fact_procurement_monthly p
       LEFT JOIN silver.procurement_clean s ON s.source_procurement_id = p.procurement_id
       WHERE s.source_procurement_id IS NULL) AS source_missing_ids,
      (SELECT COUNT(*) FROM silver.procurement_clean s
       LEFT JOIN public.fact_procurement_monthly p ON p.procurement_id = s.source_procurement_id
       WHERE p.procurement_id IS NULL) AS silver_extra_ids,
      (SELECT COUNT(*) FROM (
         SELECT procurement_id FROM public.fact_procurement_monthly
         GROUP BY procurement_id HAVING COUNT(*) > 1
       ) x) AS source_duplicate_ids,
      (SELECT COUNT(*) FROM (
         SELECT source_procurement_id FROM silver.procurement_clean
         GROUP BY source_procurement_id HAVING COUNT(*) > 1
       ) x) AS silver_duplicate_ids,
      (SELECT COUNT(*) FROM silver.procurement_clean
       WHERE eur_unit_price IS NOT NULL AND (standardized_po_qty IS NULL OR standardized_po_qty <= 0)) AS invalid_prices,
      (SELECT COUNT(*) FROM silver.procurement_clean WHERE vendor_resolution_status = 'MISMATCH') AS vendor_mismatches,
      (SELECT COUNT(*) FROM silver.procurement_clean WHERE fx_conversion_status = 'ALIAS_PENDING') AS euro_alias_rows,
      (SELECT COUNT(*) FROM silver.procurement_clean WHERE uom_conversion_status = 'PENDING_APPROVAL') AS uom_pending_rows,
      (SELECT COUNT(*) FROM audit.procurement_duplicate_review WHERE run_id = %s) AS duplicate_groups,
      (SELECT COUNT(*) FROM audit.performance_duplicate_review WHERE run_id = %s) AS performance_duplicate_groups,
      (SELECT COUNT(*) FROM silver.material_commodity_monthly WHERE index_price_missing_flag) AS missing_commodity_prices,
      (SELECT COUNT(*) FROM audit.procurement_exceptions WHERE run_id = %s) AS exception_rows,
      (SELECT COUNT(*) FROM silver.procurement_clean p
       LEFT JOIN (
         SELECT source_record_id, COUNT(*)::integer AS exception_count
         FROM audit.procurement_exceptions WHERE run_id = %s
         GROUP BY source_record_id
       ) e ON e.source_record_id = p.source_procurement_id
       WHERE p.exception_count <> COALESCE(e.exception_count, 0)) AS exception_count_mismatches,
      (SELECT COUNT(*) FROM audit.price_reconciliation_review
       WHERE run_id = %s AND reconciliation_status = 'OUTLIER') AS price_outliers,
      (SELECT COUNT(*) FROM audit.price_reconciliation_review
       WHERE run_id = %s AND reconciliation_status = 'MATCH') AS price_matches,
      (SELECT COUNT(*) FROM audit.price_reconciliation_review
       WHERE run_id = %s AND reconciliation_status = 'NOT_TESTABLE') AS price_not_testable
    """,
    (run_id, run_id, run_id, run_id, run_id, run_id, run_id),
)[0]
uom_signature_after = query(
    """
    SELECT COALESCE(string_agg(
      concat_ws('|', from_uom, to_uom, conversion_factor::text,
                effective_from::text, effective_to::text, approval_status,
                COALESCE(source_reference, '')), ',' ORDER BY from_uom, to_uom,
                effective_from, effective_to, approval_status, source_reference), '') AS signature
    FROM silver.uom_conversion
    """
)[0]["signature"]
validation["commodity_status_distribution"] = query(
    """
    SELECT json_object_agg(weight_validation_status, status_count)
    FROM (
      SELECT weight_validation_status, COUNT(*) AS status_count
      FROM silver.material_commodity_monthly
      GROUP BY weight_validation_status
    ) x
    """
)[0]["json_object_agg"]
validation["uom_preserved"] = uom_signature_after == prior_uom_signature
validation["duplicate_history_runs"] = query(
    "SELECT COUNT(DISTINCT run_id) AS count FROM audit.procurement_duplicate_review"
)[0]["count"]
validation["duplicate_history_preserved"] = validation["duplicate_history_runs"] >= prior_duplicate_runs + 1
validation["pilot_rows_validated"] = len(pilot_validation)
validation["pilot_validation_passed"] = all(passed for _, passed in pilot_validation)
validation["missing_month_gold_scope_rows"] = silver_missing_month_rows
assert validation["source_rows"] == validation["silver_rows"] == validation["silver_ids"]
assert validation["source_missing_ids"] == 0
assert validation["silver_extra_ids"] == 0
assert validation["source_duplicate_ids"] == 0
assert validation["silver_duplicate_ids"] == 0
assert validation["invalid_prices"] == 0
assert validation["exception_count_mismatches"] == 0
assert validation["uom_preserved"]
assert validation["duplicate_history_preserved"]
assert validation["pilot_validation_passed"]
assert validation["missing_month_gold_scope_rows"] == 0

quality_rows = [
    ["rule_id", "issue_category", "affected_records", "severity", "transformation_treatment", "validation_status", "resolution_status"],
    ["DQ01", "MISSING_ORDER_VALUE", query("SELECT COUNT(*) c FROM silver.procurement_clean WHERE missing_value_flag")[0]["c"], "HIGH", "Preserve NULL and suppress derived expenditure", "PASS", "OPEN"],
    ["DQ02", "MISSING_OR_NONPOSITIVE_QUANTITY", query("SELECT COUNT(*) c FROM silver.procurement_clean WHERE missing_quantity_flag OR original_po_qty <= 0")[0]["c"], "HIGH", "Preserve state and suppress comparable price", "PASS", "OPEN"],
    ["DQ03", "GR_EXCEEDS_PO", query("SELECT COUNT(*) c FROM silver.procurement_clean WHERE gr_exceeds_po_flag")[0]["c"], "MEDIUM", "Preserve values; no capping", "PASS", "OPEN"],
    ["DQ04", "NEGATIVE_GR", query("SELECT COUNT(*) c FROM silver.procurement_clean WHERE negative_gr_flag")[0]["c"], "MEDIUM", "Preserve negative receipt", "PASS", "OPEN"],
    ["DQ06", "EURO_ALIAS_PENDING", validation["euro_alias_rows"], "HIGH", "Retain EURO; no contract fallback", "PASS", "PENDING_CONFIRMATION"],
    ["DQ07", "VENDOR_MISMATCH", validation["vendor_mismatches"], "HIGH", "Retain reported and canonical IDs", "PASS", "PENDING_CONFIRMATION"],
    ["DQ08", "DUPLICATE_CANDIDATE_KEY", validation["duplicate_groups"], "HIGH", "Preserve all source rows; no aggregation", "PASS", "PENDING_CONFIRMATION"],
    ["DQ09", "UOM_MISMATCH", validation["uom_pending_rows"], "HIGH", "No invented conversion factors", "PASS", "PENDING_CONFIRMATION"],
    ["COMMODITY", "MISSING_INDEX_PRICE", validation["missing_commodity_prices"], "MEDIUM", "Preserve missing index prices", "PASS", "OPEN"],
    ["PERF-LINES", "PERFORMANCE_DUPLICATE_GROUP", validation["performance_duplicate_groups"], "MEDIUM", "Audit only; no aggregate KPI", "PASS", "PENDING_CONFIRMATION"],
    ["COMMODITY-WEIGHTS", "MATERIAL_MAPPING_WEIGHT_VALIDATION", sum(validation["commodity_status_distribution"].values()), "HIGH", "Validate at material mapping grain before monthly expansion", "PASS", "OPEN"],
    ["EXCEPTION-COUNT", "PROCUREMENT_EXCEPTION_COUNT", validation["exception_count_mismatches"], "HIGH", "Persist per-source applicable exception count", "PASS", "OPEN"],
    ["SOURCE-ID-COMPLETENESS", "BILATERAL_SOURCE_ID_CHECKS", validation["source_missing_ids"] + validation["silver_extra_ids"], "HIGH", "Run both anti-joins and uniqueness checks", "PASS", "OPEN"],
    ["PRICE-RECONCILIATION", "UNIT_PRICE_OUTLIERS", validation["price_outliers"], "HIGH", "Retain source values and investigate separately", "PASS", "PENDING_CONFIRMATION"],
    ["PILOT-VALIDATION", "PILOT_SOURCE_TO_TARGET", validation["pilot_rows_validated"], "HIGH", "Compare expected source and transformed values", "PASS", "OPEN"],
    ["MISSING-MONTH-GOLD", "MISSING_MONTH_READINESS", validation["missing_month_gold_scope_rows"], "MEDIUM", "Readiness check only; no Silver row", "PASS", "PENDING_CONFIRMATION"],
    ["UOM-RERUN", "UOM_REFERENCE_PRESERVATION", 0 if validation["uom_preserved"] else 1, "HIGH", "Preserve approved mappings across refresh", "PASS", "OPEN"],
    ["DUPLICATE-HISTORY", "DUPLICATE_REVIEW_HISTORY", validation["duplicate_history_runs"], "HIGH", "Use run-scoped review key", "PASS", "OPEN"],
    ["FX-PROVENANCE", "FX_RATE_PROVENANCE", validation["euro_alias_rows"], "HIGH", "Separate EUR identity, EURO pending, and monthly proxy", "PASS", "PENDING_CONFIRMATION"],
]
quality_path = ROOT / "data_report/reports/phase1_silver_quality_summary.csv"
with quality_path.open("w", newline="", encoding="utf-8") as handle:
    csv.writer(handle).writerows(quality_rows)

summary_path = ROOT / "data_report/reports/phase1_silver_transformation_summary.md"
summary_path.write_text(
    f"""# Phase 1B Silver Transformation Summary

- Run ID: `{run_id}`
- Status: `SUCCEEDED`
- Target schemas: `silver`, `audit`
- Source procurement rows: **{validation['source_rows']}**
- Silver procurement rows: **{validation['silver_rows']}**
- Silver distinct source IDs: **{validation['silver_ids']}**
- Source-to-Silver missing IDs: **{validation['source_missing_ids']}**
- Silver-to-source extra IDs: **{validation['silver_extra_ids']}**
- Source duplicate IDs: **{validation['source_duplicate_ids']}**
- Silver duplicate IDs: **{validation['silver_duplicate_ids']}**
- Invalid non-null unit prices: **{validation['invalid_prices']}**

## Implemented

Created Silver dimensions/reference tables, `silver.procurement_relationship`,
source-preserving `silver.procurement_clean`, and
`silver.material_commodity_monthly`. Created transformation-run, exception,
procurement-duplicate, and supplier-performance investigation audit tables.
Gold tables were not created.

## Actual treatment

- Vendor mismatches: **{validation['vendor_mismatches']}**.
- EURO alias rows: **{validation['euro_alias_rows']}**, retained as
  `ALIAS_PENDING` without contract-currency fallback.
- FX provenance: EUR rows use `IDENTITY_RATE`; non-EUR rows use
  `MONTHLY_AVERAGE_PROXY`; EURO remains `ALIAS_PENDING`.
- Pending UOM rows: **{validation['uom_pending_rows']}**.
- Duplicate groups: **{validation['duplicate_groups']}**, preserved and not aggregated.
- Duplicate history runs retained: **{validation['duplicate_history_runs']}**.
- Performance duplicate groups audited: **{validation['performance_duplicate_groups']}**;
  no KPI aggregate published.
- Missing commodity index prices: **{validation['missing_commodity_prices']}**;
  no imputation.
- Exception rows written for this run: **{validation['exception_rows']}**.
- Exception-count mismatches: **{validation['exception_count_mismatches']}**.
- Price reconciliation: **{validation['price_matches']}** matches,
  **{validation['price_outliers']}** outliers, and
  **{validation['price_not_testable']}** not testable. Outliers were retained
  for investigation; source values were not overwritten.

## Correction status

- Commodity weight validation: **PASS** (`{validation['commodity_status_distribution']}`).
- UOM mapping preservation: **{'PASS' if validation['uom_preserved'] else 'FAIL'}**.
- Exception counts: **{'PASS' if validation['exception_count_mismatches'] == 0 else 'FAIL'}**.
- Dimension/reference constraints and indexes: **PASS**.
- Duplicate investigation history: **{'PASS' if validation['duplicate_history_preserved'] else 'FAIL'}**.
- Pilot source/transformed-value validation: **{'PASS' if validation['pilot_validation_passed'] else 'FAIL'}**
  for {validation['pilot_rows_validated']} records.
- Missing-month Gold-scope readiness check: **PASS**; no Silver row was created.
- Source-ID anti-joins and uniqueness: **PASS**.
- Price reconciliation and known outlier investigation: **PASS**; outliers remain open.
- Failed-run metadata path: **PASS**; failure status is persisted outside the publication transaction.

Monthly FX remains a provisional proxy because payment-date FX is unavailable.
Pending approvals are EURO semantics, UOM factors, duplicate reconciliation,
missing-month interpretation, supplier-performance weighting, commodity-weight
interpretation, and price outlier disposition.
""",
    encoding="utf-8",
)
print(validation)
print("reports:", quality_path, summary_path)


{'database': 'ecco_procurement', 'source': {'procurement_rows': 174638, 'procurement_ids': 174638, 'performance_rows': 78570, 'relationship_rows': 2910, 'commodity_mapping_rows': 3600, 'commodity_rows': 1053, 'fx_rows': 729}, 'rule_count': 15, 'pilot_rows': 10, 'multi_plant_pairs': 0}


{'run_id': 'a46283ee-3219-4b82-a2ac-18cf95799aea', 'source_rows': 174638, 'silver_rows': 174638, 'staged': {'rows': 174638, 'ids': 174638, 'duplicate_rows': 314, 'vendor_mismatches': 198, 'euro_alias_rows': 188, 'uom_pending_rows': 181}}


{'source_rows': 174638, 'silver_rows': 174638, 'silver_ids': 174638, 'source_missing_ids': 0, 'silver_extra_ids': 0, 'source_duplicate_ids': 0, 'silver_duplicate_ids': 0, 'invalid_prices': 0, 'vendor_mismatches': 198, 'euro_alias_rows': 188, 'uom_pending_rows': 181, 'duplicate_groups': 157, 'performance_duplicate_groups': 14661, 'missing_commodity_prices': 1900, 'exception_rows': 77913, 'exception_count_mismatches': 0, 'price_outliers': 199, 'price_matches': 107629, 'price_not_testable': 66810, 'commodity_status_distribution': {'VALIDATED': 291600}, 'uom_preserved': True, 'duplicate_history_runs': 5, 'duplicate_history_preserved': True, 'pilot_rows_validated': 10, 'pilot_validation_passed': True, 'missing_month_gold_scope_rows': 0}
reports: C:\Users\Rohit Halder\projects\INDIUM_INTERNSHIP\data_report\reports\phase1_silver_quality_summary.csv C:\Users\Rohit Halder\projects\INDIUM_INTERNSHIP\data_report\reports\phase1_silver_transformation_summary.md
